# HOXC6_KO vs WT — Detailed QC Summary v2
캐시 우회용 새 버전입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, re, json, zipfile, shutil, os
ROOT=Path('/content/drive/MyDrive/HOXC6_KO_variant_analysis')
OUT=ROOT/'QC_SUMMARY'; OUT.mkdir(exist_ok=True)
SNP=ROOT/'SNP_unique_vs_WT'/'HOXC6_KO_unique_SNP_vs_WT.hg38_multianno.tsv.gz'
INDEL=ROOT/'INDEL_unique_vs_WT'/'HOXC6_KO_unique_INDEL_vs_WT.hg38_multianno.tsv.gz'
OLDZIP=ROOT/'HOXC6_KO_unique_variants_vs_WT_hg38.zip'
print('SNP',SNP.exists(),'INDEL',INDEL.exists(),'OLDZIP',OLDZIP.exists())


In [ ]:
def norm(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())
def pick(cols,names):
    d={norm(c):c for c in cols}
    for n in names:
        if norm(n) in d:return d[norm(n)]
    for c in cols:
        for n in names:
            if norm(n) and norm(n) in norm(c): return c
    return None
CANDS={'filter':['FILTER','FT'],'qual':['QUAL'],'dp':['DP','Depth'],'gq':['GQ'],'qd':['QD'],'fs':['FS'],'sor':['SOR'],'mq':['MQ'],
       'func':['Func.refGene','Func.ensGene','Func'],'exfunc':['ExonicFunc.refGene','ExonicFunc.ensGene','ExonicFunc'],
       'gene':['Gene.refGene','Gene.ensGene','Gene'],'chr':['CHROM','Chr'],'svtype':['SVType','Type'],'cnvtype':['CNVType'],'copynumber':['CopyNumber']}
def cmap(cols): return {k:pick(cols,v) for k,v in CANDS.items()}


In [ ]:
def analyze_small(path,label):
    tot=0; out={k:0 for k in ['PASS','high_conf','coding_splicing','exonic','nonsyn','frameshift','stopgain','stoploss','Tier1','Tier2','Tier3']}
    genes={}; chroms={}; cm=None; criteria=[]
    for x in pd.read_csv(path,sep='\t',compression='gzip',dtype=str,chunksize=50000,low_memory=False):
        if cm is None:
            cm=cmap(x.columns); print(label,'mapping=',cm)
        tot+=len(x)
        if cm['filter']:
            f=x[cm['filter']].fillna('').str.upper(); pm=f.isin(['PASS','.',''])|f.str.contains('PASS',regex=False)
        else: pm=pd.Series(True,index=x.index)
        hc=pm.copy(); used=[]
        for key,op,val,txt in [('qual','ge',30,'QUAL>=30'),('dp','ge',10,'DP>=10'),('gq','ge',20,'GQ>=20'),('qd','ge',2,'QD>=2'),('fs','le',60,'FS<=60'),('sor','le',3,'SOR<=3'),('mq','ge',40,'MQ>=40')]:
            c=cm[key]
            if c:
                z=pd.to_numeric(x[c],errors='coerce')
                hc &= (z>=val) if op=='ge' else (z<=val); used.append(txt)
        criteria=used
        func=x[cm['func']].fillna('').str.lower() if cm['func'] else pd.Series('',index=x.index)
        ex=x[cm['exfunc']].fillna('').str.lower() if cm['exfunc'] else pd.Series('',index=x.index)
        exonic=func.str.contains('exonic',regex=False); splice=func.str.contains('splic',regex=False); coding=exonic|splice
        nonsyn=ex.str.contains('nonsynonymous',regex=False); frames=ex.str.contains('frameshift',regex=False); sg=ex.str.contains('stopgain',regex=False); sl=ex.str.contains('stoploss',regex=False)
        damaging=nonsyn|frames|sg|sl
        out['PASS']+=int(pm.sum()); out['high_conf']+=int(hc.sum()); out['coding_splicing']+=int(coding.sum()); out['exonic']+=int(exonic.sum())
        out['nonsyn']+=int(nonsyn.sum()); out['frameshift']+=int(frames.sum()); out['stopgain']+=int(sg.sum()); out['stoploss']+=int(sl.sum())
        out['Tier1']+=int((hc&damaging).sum()); out['Tier2']+=int((hc&coding&~damaging).sum()); out['Tier3']+=int((hc&~coding).sum())
        if cm['gene']:
            for s in x.loc[hc,cm['gene']].dropna().astype(str):
                for g in re.split(r'[;,]',s):
                    g=g.strip()
                    if g and g!='.': genes[g]=genes.get(g,0)+1
        if cm['chr']:
            for k,v in x.loc[hc,cm['chr']].value_counts().items(): chroms[str(k)]=chroms.get(str(k),0)+int(v)
    return {'label':label,'total':tot,'stats':out,'criteria':criteria,'mapping':cm,'top_genes':sorted(genes.items(),key=lambda z:z[1],reverse=True)[:30],'chroms':chroms}
S=analyze_small(SNP,'SNP'); I=analyze_small(INDEL,'INDEL')


In [ ]:
# SV/CNV: 기존 combined ZIP에서 결과 파일을 찾아 기본/유형 분포 집계
TMP=Path('/content/qc_old'); shutil.rmtree(TMP,ignore_errors=True); TMP.mkdir()
sv={'total':656}; cnv={'total':1181}
if OLDZIP.exists():
    with zipfile.ZipFile(OLDZIP) as z: z.extractall(TMP)
    files=list(TMP.rglob('*'))
    svf=next((p for p in files if 'SV_events' in p.name and p.suffix=='.gz'),None)
    cnvf=next((p for p in files if 'CNV_exact' in p.name and p.suffix=='.gz'),None)
    if svf:
        d=pd.read_csv(svf,sep='\t',compression='gzip',dtype=str,low_memory=False); m=cmap(d.columns); sv['total']=len(d); sv['mapping']=m
        if m['svtype']: sv['type_counts']=d[m['svtype']].value_counts(dropna=False).to_dict()
        if m['gene']: sv['gene_overlap']=int((d[m['gene']].fillna('').astype(str).str.strip().isin(['','.'])==False).sum())
    if cnvf:
        d=pd.read_csv(cnvf,sep='\t',compression='gzip',dtype=str,low_memory=False); m=cmap(d.columns); cnv['total']=len(d); cnv['mapping']=m
        if m['cnvtype']: cnv['type_counts']=d[m['cnvtype']].value_counts(dropna=False).to_dict()
        if m['copynumber']:
            cp=pd.to_numeric(d[m['copynumber']],errors='coerce'); cnv['high_amplitude']=int(((cp<=1)|(cp>=4)).sum())
        if m['gene']: cnv['gene_overlap']=int((d[m['gene']].fillna('').astype(str).str.strip().isin(['','.'])==False).sum())
print('SV',sv); print('CNV',cnv)


In [ ]:
rows=[]
for R in [S,I]:
    s=R['stats']; t=R['total']
    rows.append({'Variant':R['label'],'KO_specific_total':t,'PASS':s['PASS'],'High_confidence':s['high_conf'],'High_conf_%':round(100*s['high_conf']/t,2),
                 'Coding_splicing':s['coding_splicing'],'Nonsynonymous':s['nonsyn'],'Frameshift':s['frameshift'],'Stopgain':s['stopgain'],'Stoploss':s['stoploss'],
                 'Tier1':s['Tier1'],'Tier2':s['Tier2'],'Tier3':s['Tier3']})
rows.append({'Variant':'SV','KO_specific_total':sv['total'],'PASS':np.nan,'High_confidence':np.nan})
rows.append({'Variant':'CNV','KO_specific_total':cnv['total'],'PASS':np.nan,'High_confidence':cnv.get('high_amplitude',np.nan)})
summary=pd.DataFrame(rows)
display(summary)

with pd.ExcelWriter(OUT/'HOXC6_KO_variant_QC_summary.xlsx',engine='openpyxl') as w:
    summary.to_excel(w,'Summary',index=False)
    pd.DataFrame(S['top_genes'],columns=['Gene','High_conf_count']).to_excel(w,'SNP_top_genes',index=False)
    pd.DataFrame(I['top_genes'],columns=['Gene','High_conf_count']).to_excel(w,'INDEL_top_genes',index=False)
    pd.DataFrame(list(S['chroms'].items()),columns=['Chr','High_conf_count']).to_excel(w,'SNP_chr',index=False)
    pd.DataFrame(list(I['chroms'].items()),columns=['Chr','High_conf_count']).to_excel(w,'INDEL_chr',index=False)
    pd.DataFrame(list(sv.get('type_counts',{}).items()),columns=['SVType','Count']).to_excel(w,'SV_types',index=False)
    pd.DataFrame(list(cnv.get('type_counts',{}).items()),columns=['CNVType','Count']).to_excel(w,'CNV_types',index=False)

payload={'SNP':S,'INDEL':I,'SV':sv,'CNV':cnv}
(OUT/'variant_qc_summary.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

lines=['HOXC6_KO vs WT Detailed QC Summary (hg38)','']
for R in [S,I]:
    s=R['stats']; t=R['total']
    lines.append(f"{R['label']}: total {t:,}")
    lines.append(f"  PASS {s['PASS']:,}; high-confidence {s['high_conf']:,} ({100*s['high_conf']/t:.2f}%)")
    lines.append(f"  coding/splicing {s['coding_splicing']:,}; nonsyn {s['nonsyn']:,}; frameshift {s['frameshift']:,}; stopgain {s['stopgain']:,}; stoploss {s['stoploss']:,}")
    lines.append(f"  Tier1 {s['Tier1']:,}; Tier2 {s['Tier2']:,}; Tier3 {s['Tier3']:,}")
    qc_text=', '.join(R['criteria']) if R['criteria'] else 'PASS only / no numeric QC columns detected'
    lines.append(f"  QC criteria used: {qc_text}")
    lines.append('')
lines.append(f"SV: {sv['total']:,} exact KO-only events")
lines.append(f"CNV: {cnv['total']:,} exact KO-only calls; high-amplitude={cnv.get('high_amplitude','NA')}")
lines.append('')
lines.append('Tier1 = high-confidence + nonsynonymous/frameshift/stopgain/stoploss')
lines.append('Tier2 = high-confidence + other coding/splicing')
lines.append('Tier3 = high-confidence non-coding')

(OUT/'SUMMARY_DETAILED.txt').write_text('\n'.join(lines),encoding='utf-8')
zipbase=ROOT/'HOXC6_KO_variant_QC_SUMMARY'
zp=Path(str(zipbase)+'.zip')
if zp.exists(): zp.unlink()
shutil.make_archive(str(zipbase),'zip',root_dir=OUT)
print('\n'.join(lines))
print('Saved:',OUT/'HOXC6_KO_variant_QC_summary.xlsx')
print('ZIP:',zp)
